In [1]:
from B0_MODEL import UNET
from B2_UTILS import (
    load_checkpoint,
    save_checkpoint,
    get_loaders,
    check_accuracy,
    save_predictions_as_imgs,
    print_and_write,
)
import torch
from tqdm import tqdm  # makes your loops show a smart progress meter
import torch.nn as nn
import torch.optim as optim
import os
import json
import numpy as np

import splitfolders
import shutil

# Test train? Train? Predict?
BATCH_1_TEST = False
TRAIN = False
MODEL_INDEX = 0
PREDICT = True

# Set training epoch munbers, early stopping epoch num
training_epoch_num = 150
early_stopping_patience = 5

# batch_size (vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162)
batch_size = 12
learning_rate = 1e-4

# ====================== Hyperparameters etc. ========================
# MAIN_PATH = "/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_with_blank"
MAIN_PATH = "/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_20240520"

MODEL_TYPES = ["secondary_strctures", "key_atoms", "residue_types"]
MAP_DIRS_TYPES = [
    ["map_sample"],
    ["map_sample", "secondary_strctures_pred_prob"],
    ["map_sample", "secondary_strctures_pred_prob", "key_atoms_pred_prob"],
]
# IN_CHANNELS = [1, 6, 10]

data_path = os.path.join(MAIN_PATH, "train_val_data")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

num_workers = 2
pin_memory = True

In [2]:
def initial_setting(model_index, data_path, map_dirs, device):
    """
    This function performs the initial setup before training process:
    1. Read the class weights of the UNET model,
    2. Read the saved epoch number,
    3. Generate paths of file that will be loaded to UNET model.
    """

    model_type = MODEL_TYPES[model_index]
    # Load the class weights of all model types
    weight_path = os.path.join(data_path, "class_weight_for_training.txt")
    try:
        # Load the weight for training
        with open(weight_path, "r") as file:
            weight_all = json.load(file)

        if model_type not in weight_all.keys():
            print(f"Weight for {model_type} is not found.")
            exit()
        else:
            c_w = weight_all[model_type]
            class_weights = torch.FloatTensor(c_w).to(device)

            # Calculate the number of output channels based on the number of tags in model_type
            out_channels = len(c_w)
            in_channels = 1
            if model_index > 0:
                for ii in range(model_index):
                    in_channels = in_channels + len(weight_all[MODEL_TYPES[ii]])

    except FileNotFoundError:
        print("Weight file does not exist.")
        exit()

    # Load the load_epoch_num data or initialize it if the file doesn't exist
    load_epoch_num_path = os.path.join(data_path, "epoch_num_path.txt")

    if not os.path.exists(load_epoch_num_path):
        load_epoch_num_all = {model_type: 0}
    else:
        with open(load_epoch_num_path, "r") as file:
            load_epoch_num_all = json.load(file)
            if model_type not in load_epoch_num_all.keys():
                load_epoch_num_all[model_type] = 0
    load_epoch_num = load_epoch_num_all[model_type]

    # Save the updated load_epoch_num_all data
    with open(load_epoch_num_path, "w") as file:
        json.dump(load_epoch_num_all, file)

    # Set file paths and make some directories when not existed
    train_map_dirs = [os.path.join(data_path, "train", map_dir) for map_dir in map_dirs]
    val_map_dirs = [os.path.join(data_path, "val", map_dir) for map_dir in map_dirs]
    train_model_dir = os.path.join(data_path, "train", model_type)
    val_model_dir = os.path.join(data_path, "val", model_type)

    load_model_path = os.path.join(
        data_path,
        "checkpoint",
        model_type,
        f"{model_type}.epoch{load_epoch_num}.pth.tar",
    )

    evl_folder_path = os.path.join(data_path, "evaluation", model_type)

    os.makedirs(os.path.dirname(load_model_path), exist_ok=True)
    os.makedirs(evl_folder_path, exist_ok=True)

    return (
        class_weights,
        in_channels,
        out_channels,
        load_epoch_num,
        train_map_dirs,
        val_map_dirs,
        train_model_dir,
        val_model_dir,
        load_model_path,
        evl_folder_path,
    )

In [3]:
model_index = 0
model_type = MODEL_TYPES[model_index]
map_dirs = MAP_DIRS_TYPES[model_index]

(
    class_weights,
    in_channels,
    out_channels,
    load_epoch_num,
    train_map_dirs,
    val_map_dirs,
    train_model_dir,
    val_model_dir,
    load_model_path,
    evl_folder_path,
) = initial_setting(model_index, data_path, map_dirs, DEVICE)

model = UNET(in_channels, out_channels).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scaler = torch.cuda.amp.GradScaler()

# Generate data loaders
train_loader, val_loader = get_loaders(
    train_map_dirs,
    train_model_dir,
    val_map_dirs,
    val_model_dir,
    batch_size,
    num_workers,
    pin_memory,
)

In [12]:
print(train_map_dirs)


['/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_20240520/train_val_data/train/map_sample']


'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST_20240520/train_val_data/train'

In [6]:
# FUN - predict - for model 0


# temp_sample_path = os.path.join(MAIN_PATH, 'ready_to_train_and_val')
# for model_type1 in MODEL_TYPES:
#     temp_sample_path1 = os.path.join(temp_sample_path, f'{model_type1}_pred_prob-test')
#     os.makedirs(temp_sample_path1, exist_ok=True)

for model_type1 in MODEL_TYPES:
    temp_sample_path = os.path.join(os.path.dirname(train_map_dirs[0]), f'{model_type1}_pred_prob')
    os.makedirs(temp_sample_path, exist_ok=True)

# pre_map_dirs = train_map_dirs
pre_map_dirs = val_map_dirs


map_names = [sorted(os.listdir(map_dir)) for map_dir in pre_map_dirs]

# print(map_names[0])
# print(map_names[1])


model.eval()
with torch.no_grad():

    count = 0

    for index in range(len(map_names[0])):
        count+=1
        print(f'{count}/{len(map_names[0])}')
        # for index in range(2):
        for i in range(len(pre_map_dirs)):
            map_dir = pre_map_dirs[i]
            map_name = map_names[i][index]
            map_path = os.path.join(map_dir, map_name)
            # print(map_path)
            map_part = np.load(map_path)

            # concatenate maps in different features
            if i == 0:
                # add a dimension of feature to original map
                map = np.expand_dims(map_part, axis=0)
            else:
                map = np.concatenate((map, map_part), axis=0)
        map = np.expand_dims(map, axis=0)

        # Move map to the specified device and perform prediction
        map = torch.from_numpy(map)
        # print(map.shape)
        map = map.to(DEVICE)
        # pred_prob = torch.softmax(model(map), 1)
        pred_prob = model(map)
        # print(pred_prob.shape)
        # writen = torch.cat((map[0, :, :, :, :], pred_prob[0, :, :, :, :]), dim=0)
        writen = pred_prob[0, :, :, :, :].cpu().numpy()

        # print(writen.shape)

        file_name = f"{model_type}_pred_prob.{map_name.split('.')[-2]}.npy"
        # print(file_name)
        file_path = os.path.join(temp_sample_path, f'{model_type}_pred_prob-test',
                                 file_name)
        np.save(file_path, writen)
        
model.train()



1/7724
2/7724
3/7724
4/7724
5/7724
6/7724
7/7724
8/7724
9/7724
10/7724
11/7724
12/7724
13/7724
14/7724
15/7724
16/7724
17/7724
18/7724
19/7724
20/7724
21/7724
22/7724
23/7724
24/7724
25/7724
26/7724
27/7724
28/7724
29/7724
30/7724
31/7724
32/7724
33/7724
34/7724
35/7724
36/7724
37/7724
38/7724
39/7724
40/7724
41/7724
42/7724
43/7724
44/7724
45/7724
46/7724
47/7724
48/7724
49/7724
50/7724
51/7724
52/7724
53/7724
54/7724
55/7724
56/7724
57/7724
58/7724
59/7724
60/7724
61/7724
62/7724
63/7724
64/7724
65/7724
66/7724
67/7724
68/7724
69/7724
70/7724
71/7724
72/7724
73/7724
74/7724
75/7724
76/7724
77/7724
78/7724
79/7724
80/7724
81/7724
82/7724
83/7724
84/7724
85/7724
86/7724
87/7724
88/7724
89/7724
90/7724
91/7724
92/7724
93/7724
94/7724
95/7724
96/7724
97/7724
98/7724
99/7724
100/7724
101/7724
102/7724
103/7724
104/7724
105/7724
106/7724
107/7724
108/7724
109/7724
110/7724
111/7724
112/7724
113/7724
114/7724
115/7724
116/7724
117/7724
118/7724
119/7724
120/7724
121/7724
122/7724
123/7724
1

UNET(
  (downs): ModuleList(
    (0): DoubleConv(
      (conv): Sequential(
        (0): Conv3d(1, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
      )
    )
    (1): DoubleConv(
      (conv): Sequential(
        (0): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv3d(128, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (4): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, track_running_sta

In [ ]:
# ## FUN - predict

# temp_sample_path = os.path.join(MAIN_PATH, 'ready_to_train_and_val')
# for model_type1 in MODEL_TYPES:
#     temp_sample_path1 = os.path.join(temp_sample_path, f'{model_type1}_pred_prob')
#     os.makedirs(temp_sample_path1, exist_ok=True)

# pre_map_dirs = val_map_dirs
# map_names = [sorted(os.listdir(map_dir)) for map_dir in pre_map_dirs]

# # print(map_names[0])
# # print(map_names[1])


# model.eval()
# with torch.no_grad():

#     count = 0

#     for index in range(len(map_names[0])):
#         count+=1
#         print(f'{count}/{len(map_names[0])}')
#         # for index in range(2):
#         for i in range(len(pre_map_dirs)):
#             map_dir = pre_map_dirs[i]
#             map_name = map_names[i][index]
#             map_path = os.path.join(map_dir, map_name)
#             map_part = np.load(map_path)

#             # concatenate maps in different features
#             if i == 0:
#                 # add a dimension of feature to original map
#                 map = np.expand_dims(map_part, axis=0)
#             else:
#                 map = np.concatenate((map, map_part), axis=0)
#         map = np.expand_dims(map, axis=0)

#         # Move map to the specified device and perform prediction
#         map = torch.from_numpy(map)
#         # print(map.shape)
#         map = map.to(DEVICE)
#         pred_prob = torch.softmax(model(map), 1)
#         # print(pred_prob.shape)
#         # writen = torch.cat((map[0, :, :, :, :], pred_prob[0, :, :, :, :]), dim=0)
#         writen = pred_prob[0, :, :, :, :].cpu().numpy()

#         # print(writen.shape)

#         file_name = f"{model_type}_pred_prob.{map_name.split('.')[-2]}.npy"
#         # print(file_name)
#         file_path = os.path.join(temp_sample_path, f'{model_type}_pred_prob',
#                                  file_name)
#         np.save(file_path, writen)
        
# model.train()

In [8]:
# split data into training and testing dataset
import splitfolders
import shutil


sample_path = os.path.join(MAIN_PATH, "train_val_data")
# sample_path = os.path.join(HOME_PATH, "TEST", "train_val_data", "test")

os.makedirs(sample_path, exist_ok=True)
splitfolders.ratio(input=temp_sample_path,
                   output=sample_path,
                   seed=44,
                   ratio=(.8, .2),
                   group_prefix=None,
                   move=True)
shutil.rmtree(temp_sample_path)

Copying files: 30896 files [04:52, 105.67 files/s]


In [ ]:
# test generatred data
import numpy as np
import matplotlib.pyplot as plt

for num in range(0, 5):
    model_type = 'map_sample'
    mapname = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/map.{num}.npy'
    aaa = np.load(mapname)
    # print(np.max(aaa))
    plt.imshow(aaa[:, :, 30], cmap='gray', origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()

    model_type = 'secondary_strctures'
    filename = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/model_{model_type}.{num}.npy'
    aaa = np.load(filename)
    plt.imshow(aaa[:, :, 30], origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()

    model_type = 'secondary_strctures_pred_prob'
    filename = f'/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST/train_val_data/train/{model_type}/{model_type}.{num}.npy'
    aaa = np.load(filename)
    plt.imshow(aaa[:, :, 30], origin='lower')
    plt.title(f'{model_type}.{num}')
    plt.show()
    print("###")